**Faker usage example: demo**

This notebook shows the main parts of the project. It does not change the database.

Run `python main.py` first: it fills the database with data. Before that, start PostgreSQL (`docker compose up -d`) and fill in `.env` (see the README).

The notebook has two parts:

1. how the fake data is generated, on a small example without a database
2. example queries to the database that `main.py` has filled

The table structure is inspired by [this Codewars task](https://www.codewars.com/kata/5818bde9559ff58bd90004a2/sql) (table names and some column types are adapted). The task itself is not solved here.

**Table of contents**<a id='toc0_'></a>    
- [Generate demo data (no database)](#toc1_)    
  - [Actors](#toc1_1_)    
  - [Films](#toc1_2_)    
  - [Film-actor links](#toc1_3_)    
  - [Same seed, same data](#toc1_4_)    
- [Connect to the database](#toc2_)    
  - [Tables](#toc2_1_)    
- [Look at the data](#toc3_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

In [1]:
import pandas as pd
from sqlalchemy import inspect, text

import main
from src import db, generate

# <a id='toc1_'></a>[Generate demo data (no database)](#toc0_)

`make_faker()` creates a Faker object with the custom `film_genre()` provider. The seed is stored inside this object, so the same seed gives the same data. The sizes here are small, `main.py` uses larger ones.

In [2]:
fake = generate.make_faker(main.SEED)

The custom provider adds the method `film_genre()`:

In [3]:
generate.make_faker().film_genre()

'Comedy'

## <a id='toc1_1_'></a>[Actors](#toc0_)

In [4]:
actors = generate.make_actors(fake, 5)
actors

,first_name,last_name
actor_id,,
1,Danielle,Rhodes
2,Angel,Mcclain
3,Joshua,Miller
4,Jeffrey,Henderson
5,Jill,Johnson


## <a id='toc1_2_'></a>[Films](#toc0_)

In [5]:
films = generate.make_films(fake, 5)
films

,title,genre,rating,release_date
film_id,,,,
1,Despite Measure Ago,Comedy,5.5,2021-04-07
2,Current Practice Nation,Thriller,5.4,2026-06-29
3,Determine Operation Speak,Thriller,8.7,1998-06-30
4,According South Recently,Drama,4.3,2015-10-05
5,Future Choice Whatever,Thriller,1.5,1995-05-13


## <a id='toc1_3_'></a>[Film-actor links](#toc0_)

Pairs (film, actor) are drawn at random. Duplicate pairs are dropped, so there can be fewer pairs than draws.

In [6]:
links = generate.make_films_actors(fake, films, actors, 10)
print(f"{len(links)} pairs from 10 draws")
links.reset_index()

8 pairs from 10 draws


,film_id,actor_id
0,3,2
1,5,1
2,4,2
3,5,4
4,1,2
5,2,2
6,2,5
7,1,4


## <a id='toc1_4_'></a>[Same seed, same data](#toc0_)

Two Faker objects with the same seed produce exactly the same data:

In [7]:
first = generate.make_actors(generate.make_faker(main.SEED), 5)
second = generate.make_actors(generate.make_faker(main.SEED), 5)
first.equals(second)

True

# <a id='toc2_'></a>[Connect to the database](#toc0_)

`db.get_engine()` reads the connection settings from `.env` and creates a SQLAlchemy engine. It connects on the first query.

In [8]:
engine = db.get_engine()

## <a id='toc2_1_'></a>[Tables](#toc0_)

The tables are created by `db.apply_schema()` from `sql/schema.sql`:

In [9]:
insp = inspect(engine)
insp.get_table_names()

['actors', 'films', 'films_actors']

In [10]:
pd.DataFrame(insp.get_columns("films"))[["name", "type", "nullable"]]

,name,type,nullable
0,film_id,INTEGER,False
1,title,VARCHAR(255),False
2,genre,VARCHAR(50),True
3,rating,"NUMERIC(3, 1)",True
4,release_date,DATE,True


The rating has a CHECK constraint (from 1.0 to 10.0):

In [11]:
insp.get_check_constraints("films")

[{'name': 'films_rating_check',
  'sqltext': 'rating >= 1.0 AND rating <= 10.0',
  'comment': None}]

# <a id='toc3_'></a>[Look at the data](#toc0_)

A small helper that runs a query and returns a DataFrame:

In [12]:
def query(sql: str) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

Row counts:

In [13]:
query("""--sql
SELECT 'actors' AS table_name, COUNT(*) AS row_count FROM actors
UNION ALL
SELECT 'films', COUNT(*) FROM films
UNION ALL
SELECT 'films_actors', COUNT(*) FROM films_actors
""")

,table_name,row_count
0,actors,300
1,films,100
2,films_actors,990


The five earliest films by release date:

In [14]:
query("""--sql
SELECT *
FROM films
ORDER BY release_date, film_id
LIMIT 5
""")

,film_id,title,genre,rating,release_date
0,53,My Including Could,Thriller,3.8,1990-06-01
1,98,Interesting Economic Suddenly,Romance,6.5,1990-07-11
2,45,Trouble Consider One,Documentary,7.0,1991-10-15
3,47,Girl Four Prove,Thriller,5.6,1991-12-21
4,59,Need Behavior Yeah,Thriller,6.8,1992-02-01


Average number of actors per film:

In [15]:
query("""--sql
SELECT
  ROUND(COUNT(*)::NUMERIC / (SELECT COUNT(*) FROM films), 2) AS avg_actors_per_film
FROM films_actors
""")

,avg_actors_per_film
0,9.9


Number of films and average rating by genre:

In [16]:
query("""--sql
SELECT genre, COUNT(*) AS film_count, ROUND(AVG(rating), 2) AS avg_rating
FROM films
GROUP BY genre
ORDER BY film_count DESC, genre
""")

,genre,film_count,avg_rating
0,Action,15,4.53
1,Thriller,15,4.81
2,Comedy,14,5.81
3,Romance,13,5.28
4,Horror,12,5.47
5,Documentary,11,6.28
6,Drama,11,6.07
7,Mystery,9,4.89


Films with the most actors:

In [17]:
query("""--sql
SELECT f.title, COUNT(*) AS actor_count
FROM films_actors AS fa
INNER JOIN films AS f ON f.film_id = fa.film_id
GROUP BY f.film_id, f.title
ORDER BY actor_count DESC, f.title
LIMIT 5
""")

,title,actor_count
0,These Tell Everybody,18
1,Able Late Order,15
2,Building Court Build,15
3,Develop Reality Fill,15
4,Eye Buy Read,15


In [18]:
engine.dispose()